# LMM (Titans): Mortality Prediction on MIMIC-III

This notebook demonstrates the **LMM** (Neural Long-term Memory) model for mortality prediction on MIMIC-III.

**Paper**: Ali Behrouz et al. "Titans: Learning to Memorize at Test Time." arXiv 2501.00663, 2025.

LMM uses surprise-based gradient updates with momentum to preferentially memorize unexpected inputs. For EHR data, this means rare but clinically significant events (a sudden lab spike, a new critical diagnosis) are stored more strongly in memory than routine visits.

This notebook shows two configurations:
1. **LMM standalone** — LMM as a direct sequence encoder (like RNN)
2. **GRASP + LMM** — LMM as the backbone inside GRASP's patient clustering framework

### Compute Tracking

This notebook tracks energy consumption, GPU utilization, and CO2 emissions for each training run using `codecarbon` and `pynvml`. This data supports research into the environmental cost of ML experiments and helps inform datacenter policy.

## Step 1: Load the MIMIC-III Dataset

We load the MIMIC-III dataset using PyHealth's `MIMIC3Dataset` class.

**Two configurations:**
- **Local (default):** Set `dev=True` with synthetic GCS data for quick testing
- **H200 / full run:** Set `USE_LOCAL_DATA = True` below and point `LOCAL_ROOT` to your MIMIC-III CSV directory

In [ ]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset

# ── Configuration ─────────────────────────────────────────
# Set USE_LOCAL_DATA = True on the H200 where MIMIC-III CSVs live
USE_LOCAL_DATA = False
LOCAL_ROOT = "/home/olowo2"           # directory with ADMISSIONS.csv.gz, etc.
LOCAL_CACHE = "/home/olowo2/tmp/pyhealth_cache"
# ──────────────────────────────────────────────────────────

if USE_LOCAL_DATA:
    base_dataset = MIMIC3Dataset(
        root=LOCAL_ROOT,
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=LOCAL_CACHE,
        dev=False,  # full dataset
    )
else:
    base_dataset = MIMIC3Dataset(
        root="https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III",
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=tempfile.TemporaryDirectory().name,
        dev=True,  # small synthetic subset
    )

base_dataset.stats()

## Step 2: Define the Mortality Prediction Task

The `MortalityPredictionMIMIC3` task extracts samples from the raw EHR data:
- Extracts diagnosis codes (ICD-9), procedure codes, and drug information from each visit
- Creates binary labels based on in-hospital mortality
- Filters out visits without sufficient clinical codes

In [2]:
from pyhealth.tasks import MortalityPredictionMIMIC3

task = MortalityPredictionMIMIC3()
samples = base_dataset.set_task(task)

print(f"Generated {len(samples)} samples")
print(f"\nInput schema: {samples.input_schema}")
print(f"Output schema: {samples.output_schema}")

Setting task MortalityPredictionMIMIC3 for mimic3 base dataset...
Task cache paths: task_df=/var/folders/1w/tzjkjsks6q3_8tzfy9bbf4m00000gn/T/tmp_z3mcz4z/e742d19f-576a-5354-99b1-d7ebef44ced6/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/task_df.ld, samples=/var/folders/1w/tzjkjsks6q3_8tzfy9bbf4m00000gn/T/tmp_z3mcz4z/e742d19f-576a-5354-99b1-d7ebef44ced6/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 1000 patients. (Polars threads: 14)


  0%|          | 0/1000 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████| 1000/1000 [00:00<00:00, 1779.94it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...


Label mortality vocab: {0: 0, 1: 1}
Processing samples and saving to /var/folders/1w/tzjkjsks6q3_8tzfy9bbf4m00000gn/T/tmp_z3mcz4z/e742d19f-576a-5354-99b1-d7ebef44ced6/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld...
Applying processors on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 20 samples. (0 to 20)


/var/folders/1w/tzjkjsks6q3_8tzfy9bbf4m00000gn/T/ipykernel_18580/2882863153.py:4: UserWarning: A newer version of litdata is available (0.2.61). Please consider upgrading with `pip install -U litdata`. Not all functionalities of the platform can be guaranteed to work with the current version.
  samples = base_dataset.set_task(task)
  0%|          | 0/20 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1']` data format.


100%|██████████| 20/20 [00:00<00:00, 3651.67it/s]

Worker 0 finished processing samples.
Cached processed samples to /var/folders/1w/tzjkjsks6q3_8tzfy9bbf4m00000gn/T/tmp_z3mcz4z/e742d19f-576a-5354-99b1-d7ebef44ced6/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Generated 20 samples

Input schema: {'conditions': 'sequence', 'procedures': 'sequence', 'drugs': 'sequence'}
Output schema: {'mortality': 'binary'}


## Step 3: Split Data and Create Loaders

We split by patient to avoid data leakage, then create batched data loaders.

In [3]:
from pyhealth.datasets import split_by_patient, get_dataloader

train_dataset, val_dataset, test_dataset = split_by_patient(
    samples, [0.8, 0.1, 0.1]
)

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 16
Validation samples: 2
Test samples: 2


## Compute Tracking Setup

We log hardware specs, energy consumption, GPU power/utilization, and CO2 emissions.

Install dependencies if needed:
```bash
pip install codecarbon pynvml
```

**Metrics tracked per training run:**

| Metric | What it measures | Unit |
|---|---|---|
| Wall time | Real elapsed clock time | seconds |
| Energy consumed | Electricity drawn by GPU | kWh |
| CO2 emissions | Energy × grid carbon intensity | kg CO2eq |
| Peak / avg GPU power | How hard the chip works | Watts |
| GPU utilization | % time GPU cores are computing | % |
| GPU memory | VRAM allocated vs available | GB |
| Energy per epoch | Diminishing returns indicator | kWh |
| Energy per sample | Unit cost for deployment | Joules |
| Batch throughput | Samples processed per second | samples/s |

In [4]:
import time
import platform
import torch

# ── Hardware Info ──────────────────────────────────────────
print("=" * 60)
print("Hardware Information")
print("=" * 60)
print(f"Platform:     {platform.system()} {platform.machine()}")
print(f"Python:       {platform.python_version()}")
print(f"PyTorch:      {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU:          {gpu_name}")
    print(f"GPU Memory:   {gpu_mem_gb:.1f} GB")
    print(f"CUDA:         {torch.version.cuda}")
else:
    print("GPU:          None (CPU only)")
    print("  Note: Compute metrics will be limited without GPU")

# ── GPU Monitoring Helper ─────────────────────────────────
gpu_available = torch.cuda.is_available()
nvml_available = False

if gpu_available:
    try:
        import pynvml
        pynvml.nvmlInit()
        nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        nvml_available = True
        print(f"\npynvml:       initialized (live GPU monitoring enabled)")
    except (ImportError, Exception) as e:
        print(f"\npynvml:       not available ({e})")
        print("  Install with: pip install pynvml")

# ── Carbon Tracking Helper ────────────────────────────────
codecarbon_available = False
try:
    from codecarbon import EmissionsTracker
    codecarbon_available = True
    print(f"codecarbon:   available (CO2 tracking enabled)")
except ImportError:
    print(f"codecarbon:   not available")
    print("  Install with: pip install codecarbon")


class ComputeTracker:
    """Tracks energy, GPU metrics, and timing for a training run."""

    def __init__(self, name, num_samples, num_epochs):
        self.name = name
        self.num_samples = num_samples
        self.num_epochs = num_epochs
        self.power_readings = []
        self.util_readings = []
        self.mem_readings = []

    def start(self):
        self.start_time = time.time()
        if gpu_available:
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        if codecarbon_available:
            self.tracker = EmissionsTracker(
                project_name=self.name,
                log_level="error",
                save_to_file=False,
            )
            self.tracker.start()
        self._sample_gpu()
        return self

    def _sample_gpu(self):
        if nvml_available:
            power = pynvml.nvmlDeviceGetPowerUsage(nvml_handle) / 1000  # mW -> W
            util = pynvml.nvmlDeviceGetUtilizationRates(nvml_handle).gpu
            mem_info = pynvml.nvmlDeviceGetMemoryInfo(nvml_handle)
            self.power_readings.append(power)
            self.util_readings.append(util)
            self.mem_readings.append(mem_info.used / 1e9)

    def sample(self):
        """Call periodically during training to collect GPU readings."""
        self._sample_gpu()

    def stop(self):
        if gpu_available:
            torch.cuda.synchronize()
        self.wall_time = time.time() - self.start_time
        self._sample_gpu()

        self.emissions_data = None
        if codecarbon_available:
            self.tracker.stop()
            self.emissions_data = self.tracker.final_emissions_data

        return self.report()

    def report(self):
        r = {"name": self.name}
        r["wall_time_s"] = self.wall_time
        r["wall_time_min"] = self.wall_time / 60
        r["num_epochs"] = self.num_epochs
        r["num_samples"] = self.num_samples

        # Throughput
        total_samples_processed = self.num_samples * self.num_epochs
        r["batch_throughput_sps"] = total_samples_processed / self.wall_time
        r["time_per_epoch_s"] = self.wall_time / max(self.num_epochs, 1)

        # Energy (from codecarbon)
        if self.emissions_data:
            r["energy_kwh"] = self.emissions_data.energy_consumed
            r["co2_kg"] = self.emissions_data.emissions
            r["energy_per_epoch_kwh"] = r["energy_kwh"] / max(self.num_epochs, 1)
            r["energy_per_sample_j"] = (r["energy_kwh"] * 3.6e6) / max(total_samples_processed, 1)
        else:
            r["energy_kwh"] = None
            r["co2_kg"] = None

        # GPU metrics (from pynvml)
        if self.power_readings:
            r["peak_power_w"] = max(self.power_readings)
            r["avg_power_w"] = sum(self.power_readings) / len(self.power_readings)
            r["avg_gpu_util_pct"] = sum(self.util_readings) / len(self.util_readings)
            r["peak_gpu_mem_gb"] = max(self.mem_readings)
        elif gpu_available:
            r["peak_gpu_mem_gb"] = torch.cuda.max_memory_allocated() / 1e9

        return r


def print_report(r):
    print(f"\n{'=' * 50}")
    print(f"Compute Report: {r['name']}")
    print(f"{'=' * 50}")
    print(f"Wall time:          {r['wall_time_s']:.1f}s ({r['wall_time_min']:.2f} min)")
    print(f"Epochs:             {r['num_epochs']}")
    print(f"Time per epoch:     {r['time_per_epoch_s']:.2f}s")
    print(f"Throughput:         {r['batch_throughput_sps']:.1f} samples/s")

    if r.get("energy_kwh") is not None:
        print(f"\nEnergy consumed:    {r['energy_kwh']:.6f} kWh")
        print(f"CO2 emissions:      {r['co2_kg']:.6f} kg CO2eq")
        print(f"Energy per epoch:   {r['energy_per_epoch_kwh']:.6f} kWh")
        print(f"Energy per sample:  {r['energy_per_sample_j']:.4f} J")

    if r.get("peak_power_w"):
        print(f"\nPeak GPU power:     {r['peak_power_w']:.0f} W")
        print(f"Avg GPU power:      {r['avg_power_w']:.0f} W")
        print(f"Avg GPU utilization: {r['avg_gpu_util_pct']:.0f}%")

    if r.get("peak_gpu_mem_gb"):
        total_gb = gpu_mem_gb if gpu_available else 0
        used = r["peak_gpu_mem_gb"]
        print(f"Peak GPU memory:    {used:.2f} GB / {total_gb:.0f} GB ({used/max(total_gb,1)*100:.0f}%)")

    print(f"{'=' * 50}")


print("\nComputeTracker ready.")

Hardware Information
Platform:     Darwin arm64
Python:       3.12.12
PyTorch:      2.7.1
GPU:          None (CPU only)
  Note: Compute metrics will be limited without GPU
codecarbon:   available (CO2 tracking enabled)

ComputeTracker ready.


## Step 4: LMM Standalone Model

First, we train the LMM model as a standalone sequence encoder — the same role as RNN, but using surprise-based memory instead of fixed gates.

### Key Parameters:
- `embedding_dim`: Dimension of code embeddings
- `hidden_dim`: Dimension of the memory's hidden state
- `memory_depth`: Number of layers in the memory MLP (default: 2)
- `use_momentum`: Whether surprise includes momentum from past timesteps (default: True)
- `use_weight_decay`: Whether memory undergoes adaptive forgetting (default: True)

In [5]:
from pyhealth.models import LMM

# Using best hyperparams from GRASP GRU sweep:
# emb=32, hid=32, cluster=8, lr=5e-4, wd=1e-4
lmm_model = LMM(
    dataset=samples,
    embedding_dim=32,
    hidden_dim=32,
    memory_depth=2,
)

print(f"LMM model: {sum(p.numel() for p in lmm_model.parameters()):,} parameters")

LMM model: 43,946 parameters


In [6]:
from pyhealth.trainer import Trainer

NUM_EPOCHS = 50

lmm_trainer = Trainer(
    model=lmm_model,
    metrics=["roc_auc", "pr_auc", "accuracy", "f1"],
)

# Start compute tracking
lmm_tracker = ComputeTracker(
    "LMM-standalone", len(train_dataset), NUM_EPOCHS,
).start()

lmm_trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=NUM_EPOCHS,
    monitor="roc_auc",
    optimizer_params={"lr": 5e-4, "weight_decay": 1e-4},
)

lmm_compute = lmm_tracker.stop()
print_report(lmm_compute)

LMM(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(164, 32, padding_idx=0)
    (procedures): Embedding(75, 32, padding_idx=0)
    (drugs): Embedding(432, 32, padding_idx=0)
  ))
  (lmm): ModuleDict(
    (conditions): LMMLayer(
      (proj_k): Linear(in_features=32, out_features=32, bias=True)
      (proj_v): Linear(in_features=32, out_features=32, bias=True)
      (proj_q): Linear(in_features=32, out_features=32, bias=True)
      (memory): Sequential(
        (0): Linear(in_features=32, out_features=64, bias=True)
        (1): SiLU()
        (2): Linear(in_features=64, out_features=32, bias=True)
      )
      (gate_theta): Linear(in_features=32, out_features=1, bias=True)
      (gate_eta): Linear(in_features=32, out_features=1, bias=True)
      (gate_alpha): Linear(in_features=32, out_features=1, bias=True)
      (dropout_layer): Dropout(p=0.5, inplace=False)
    )
    (procedures): LMMLayer(
      (proj_k): Linear(in_features=32, out_fea

[codecarbon WARNING @ 00:33:52] Multiple instances of codecarbon are allowed to run at the same time.


Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.0005, 'weight_decay': 0.0001}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x108aca120>
Monitor: roc_auc
Monitor criterion: max
Epochs: 50
Patience: None



Epoch 0 / 50:   0%|          | 0/1 [00:00<?, ?it/s]

--- Train epoch-0, step-1 ---
loss: 0.6843


...

Epoch 49 / 50:   0%|          | 0/1 [00:00<?, ?it/s]

--- Train epoch-49, step-50 ---
loss: 0.6649


Evaluation: 100%|██████████| 1/1 [00:00<00:00, 77.77it/s]

--- Eval epoch-49, step-50 ---
roc_auc: nan
pr_auc: 0.0000
accuracy: 1.0000
f1: 0.0000
loss: 0.6616

Compute Report: LMM-standalone
Wall time:          8.9s (0.15 min)
Epochs:             50
Time per epoch:     0.18s
Throughput:         90.3 samples/s

Energy consumed:    0.000076 kWh
CO2 emissions:      0.000028 kg CO2eq
Energy per epoch:   0.000002 kWh
Energy per sample:  0.3425 J



/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [7]:
lmm_results = lmm_trainer.evaluate(test_dataloader)

print("=" * 50)
print("LMM Standalone — Test Set Performance")
print("=" * 50)
for metric, value in lmm_results.items():
    print(f"{metric}: {value:.4f}")

Evaluation: 100%|██████████| 1/1 [00:00<00:00, 33.30it/s]

LMM Standalone — Test Set Performance
roc_auc: nan
pr_auc: 0.0000
accuracy: 1.0000
f1: 0.0000
loss: 0.6643



/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Step 5: GRASP + LMM

Now we use LMM as the backbone inside GRASP. GRASP adds patient clustering via k-means, refines cluster representations with a GCN, and blends cluster knowledge back via a learned gate.

This tests whether surprise-based memory (LMM) and cross-patient knowledge (GRASP) are complementary:
- **LMM**: "What should I remember from *this* patient's history?"
- **GRASP**: "What can I learn from *similar* patients?"

In [ ]:
from pyhealth.models import GRASP

grasp_lmm_model = GRASP(
    dataset=samples,
    embedding_dim=32,
    hidden_dim=32,
    cluster_num=8 if USE_LOCAL_DATA else 2,  # 8 for full data, 2 for dev
    block="LMM",
)

print(f"GRASP+LMM model: {sum(p.numel() for p in grasp_lmm_model.parameters()):,} parameters")

In [9]:
grasp_lmm_trainer = Trainer(
    model=grasp_lmm_model,
    metrics=["roc_auc", "pr_auc", "accuracy", "f1"],
)

# Start compute tracking
grasp_tracker = ComputeTracker(
    "GRASP+LMM", len(train_dataset), NUM_EPOCHS,
).start()

grasp_lmm_trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=NUM_EPOCHS,
    monitor="roc_auc",
    optimizer_params={"lr": 5e-4, "weight_decay": 1e-4},
)

grasp_compute = grasp_tracker.stop()
print_report(grasp_compute)

GRASP(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(164, 32, padding_idx=0)
    (procedures): Embedding(75, 32, padding_idx=0)
    (drugs): Embedding(432, 32, padding_idx=0)
  ))
  (grasp): ModuleDict(
    (conditions): GRASPLayer(
      (backbone): LMMLayer(
        (proj_k): Linear(in_features=32, out_features=32, bias=True)
        (proj_v): Linear(in_features=32, out_features=32, bias=True)
        (proj_q): Linear(in_features=32, out_features=32, bias=True)
        (memory): Sequential(
          (0): Linear(in_features=32, out_features=64, bias=True)
          (1): SiLU()
          (2): Linear(in_features=64, out_features=32, bias=True)
        )
        (gate_theta): Linear(in_features=32, out_features=1, bias=True)
        (gate_eta): Linear(in_features=32, out_features=1, bias=True)
        (gate_alpha): Linear(in_features=32, out_features=1, bias=True)
        (dropout_layer): Dropout(p=0, inplace=False)
      )
      (relu): Re

Epoch 0 / 50:   0%|          | 0/1 [00:00<?, ?it/s]

--- Train epoch-0, step-1 ---
loss: 0.7245


Evaluation: 100%|██████████| 1/1 [00:00<00:00, 76.02it/s]

--- Eval epoch-0, step-1 ---
roc_auc: nan
pr_auc: 0.0000
accuracy: 0.0000
f1: 0.0000
loss: 0.7402



...

Epoch 49 / 50:   0%|          | 0/1 [00:00<?, ?it/s]

--- Train epoch-49, step-50 ---
loss: 0.6602


Evaluation: 100%|██████████| 1/1 [00:00<00:00, 73.56it/s]

--- Eval epoch-49, step-50 ---
roc_auc: nan
pr_auc: 0.0000
accuracy: 1.0000
f1: 0.0000
loss: 0.6551

Compute Report: GRASP+LMM
Wall time:          7.0s (0.12 min)
Epochs:             50
Time per epoch:     0.14s
Throughput:         113.9 samples/s

Energy consumed:    0.000079 kWh
CO2 emissions:      0.000029 kg CO2eq
Energy per epoch:   0.000002 kWh
Energy per sample:  0.3559 J



/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [10]:
grasp_lmm_results = grasp_lmm_trainer.evaluate(test_dataloader)

print("=" * 50)
print("GRASP + LMM — Test Set Performance")
print("=" * 50)
for metric, value in grasp_lmm_results.items():
    print(f"{metric}: {value:.4f}")

Evaluation: 100%|██████████| 1/1 [00:00<00:00, 31.21it/s]

GRASP + LMM — Test Set Performance
roc_auc: nan
pr_auc: 0.0000
accuracy: 1.0000
f1: 0.0000
loss: 0.6555



/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/Users/loewcx/Documents/root/70-79_Projects/PyHealth/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Step 6: Compare Results — Performance and Compute

Side-by-side comparison of model accuracy AND compute cost. This is the data that matters for understanding the true cost of ML research.

In [11]:
# ── Model Performance ──────────────────────────────────────
print("=" * 60)
print(f"{'MODEL PERFORMANCE':<60}")
print("=" * 60)
print(f"{'Metric':<15} {'LMM Standalone':>15} {'GRASP + LMM':>15}")
print("-" * 60)
for metric in lmm_results:
    lmm_val = lmm_results[metric]
    grasp_val = grasp_lmm_results[metric]
    better = "<-" if grasp_val > lmm_val else "->" if lmm_val > grasp_val else "=="
    print(f"{metric:<15} {lmm_val:>15.4f} {grasp_val:>15.4f}  {better}")

# ── Compute Cost ──────────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"{'COMPUTE COST':<60}")
print("=" * 60)
print(f"{'Metric':<25} {'LMM Standalone':>15} {'GRASP + LMM':>15}")
print("-" * 60)

rows = [
    ("Wall time (s)", "wall_time_s", ".1f"),
    ("Time/epoch (s)", "time_per_epoch_s", ".2f"),
    ("Throughput (samples/s)", "batch_throughput_sps", ".1f"),
]

if lmm_compute.get("energy_kwh") is not None:
    rows += [
        ("Energy (kWh)", "energy_kwh", ".6f"),
        ("CO2 (kg)", "co2_kg", ".6f"),
        ("Energy/epoch (kWh)", "energy_per_epoch_kwh", ".6f"),
        ("Energy/sample (J)", "energy_per_sample_j", ".4f"),
    ]

if lmm_compute.get("peak_power_w"):
    rows += [
        ("Peak power (W)", "peak_power_w", ".0f"),
        ("Avg power (W)", "avg_power_w", ".0f"),
        ("GPU utilization (%)", "avg_gpu_util_pct", ".0f"),
    ]

if lmm_compute.get("peak_gpu_mem_gb"):
    rows.append(("Peak GPU memory (GB)", "peak_gpu_mem_gb", ".2f"))

for label, key, fmt in rows:
    lv = lmm_compute.get(key)
    gv = grasp_compute.get(key)
    lstr = f"{lv:{fmt}}" if lv is not None else "N/A"
    gstr = f"{gv:{fmt}}" if gv is not None else "N/A"
    print(f"{label:<25} {lstr:>15} {gstr:>15}")

print("=" * 60)

MODEL PERFORMANCE                                           
Metric           LMM Standalone     GRASP + LMM
------------------------------------------------------------
roc_auc                     nan             nan  ==
pr_auc                   0.0000          0.0000  ==
accuracy                 1.0000          1.0000  ==
f1                       0.0000          0.0000  ==
loss                     0.6643          0.6555  ->

COMPUTE COST                                                
Metric                     LMM Standalone     GRASP + LMM
------------------------------------------------------------
Wall time (s)                         8.9             7.0
Time/epoch (s)                       0.18            0.14
Throughput (samples/s)               90.3           113.9
Energy (kWh)                     0.000076        0.000079
CO2 (kg)                         0.000028        0.000029
Energy/epoch (kWh)               0.000002        0.000002
Energy/sample (J)                  0.3425

## Step 7: Extract Patient Embeddings

Both models produce patient embeddings that can be used for downstream tasks like patient similarity search or cohort discovery.

In [12]:
import torch

grasp_lmm_model.eval()
test_batch = next(iter(test_dataloader))
test_batch["embed"] = True

with torch.no_grad():
    output = grasp_lmm_model(**test_batch)

print(f"Embedding shape: {output['embed'].shape}")
print(f"  - Batch size: {output['embed'].shape[0]}")
print(f"  - Embedding dim: {output['embed'].shape[1]}")

print("\nSample Predictions:")
predictions = output["y_prob"].cpu().numpy()
true_labels = output["y_true"].cpu().numpy()

for i in range(min(5, len(predictions))):
    pred = predictions[i][0]
    true = int(true_labels[i][0])
    label = "Mortality" if pred > 0.5 else "Survival"
    print(f"  Patient {i + 1}: Predicted={pred:.3f}, True={true}, -> {label}")

Embedding shape: torch.Size([2, 96])
  - Batch size: 2
  - Embedding dim: 96

Sample Predictions:
  Patient 1: Predicted=0.481, True=0, -> Survival
  Patient 2: Predicted=0.480, True=0, -> Survival


## Step 8: Compute Cost Summary for Policy

This section translates raw compute metrics into policy-relevant context. The goal is to answer: **what is the real environmental cost of this ML research?**

In [ ]:
import json
from datetime import datetime

# ── Policy Context ────────────────────────────────────────
# Reference: average US home uses ~30 kWh/day
# Reference: driving 1 mile ≈ 0.4 kg CO2
# Reference: PUE (datacenter overhead) typically 1.1-1.6x

total_energy = (lmm_compute.get("energy_kwh") or 0) + (grasp_compute.get("energy_kwh") or 0)
total_co2 = (lmm_compute.get("co2_kg") or 0) + (grasp_compute.get("co2_kg") or 0)
total_time = lmm_compute["wall_time_s"] + grasp_compute["wall_time_s"]

print("=" * 60)
print("ENVIRONMENTAL IMPACT — POLICY SUMMARY")
print("=" * 60)

print(f"\n--- This notebook's total footprint ---")
print(f"Total wall time:      {total_time:.0f}s ({total_time/60:.1f} min)")
if total_energy > 0:
    print(f"Total energy:         {total_energy:.6f} kWh")
    print(f"Total CO2:            {total_co2:.6f} kg CO2eq")

    # Analogies
    lightbulb_min = (total_energy / 0.01) * 60  # 10W bulb = 0.01 kWh/hr
    home_fraction = total_energy / 30 * 100
    miles_driven = total_co2 / 0.4

    print(f"\n--- In everyday terms ---")
    print(f"Equivalent to running a 10W LED bulb for {lightbulb_min:.0f} minutes")
    print(f"Equivalent to {home_fraction:.4f}% of a US home's daily electricity")
    if total_co2 > 0:
        print(f"Equivalent to driving {miles_driven:.3f} miles")

    # Extrapolation to full sweep
    FULL_SWEEP_CONFIGS = 108  # from sweep_grasp.py
    sweep_energy = total_energy * FULL_SWEEP_CONFIGS / 2  # /2 because we ran 2 configs
    sweep_co2 = total_co2 * FULL_SWEEP_CONFIGS / 2
    print(f"\n--- Extrapolated: full hyperparameter sweep ({FULL_SWEEP_CONFIGS} configs) ---")
    print(f"Estimated energy:     {sweep_energy:.4f} kWh")
    print(f"Estimated CO2:        {sweep_co2:.4f} kg CO2eq")
    print(f"Equivalent to driving {sweep_co2 / 0.4:.1f} miles")

# GPU efficiency
if lmm_compute.get("avg_gpu_util_pct") is not None:
    avg_util = (lmm_compute["avg_gpu_util_pct"] + grasp_compute.get("avg_gpu_util_pct", 0)) / 2
    print(f"\n--- GPU efficiency ---")
    print(f"Average GPU utilization: {avg_util:.0f}%")
    print(f"Idle energy waste:       ~{(100 - avg_util):.0f}% of GPU power went to idle/waiting")
    if lmm_compute.get("peak_gpu_mem_gb"):
        mem_used = max(lmm_compute.get("peak_gpu_mem_gb", 0), grasp_compute.get("peak_gpu_mem_gb", 0))
        total_gb = gpu_mem_gb if gpu_available else 80
        print(f"Memory utilization:      {mem_used:.1f} GB used of {total_gb:.0f} GB ({mem_used/total_gb*100:.0f}%)")
        if mem_used < total_gb * 0.1:
            print(f"  -> This model could run on a much smaller GPU (< {max(4, mem_used * 2):.0f} GB)")
            print(f"     An H200 (700W TDP) is overkill — a consumer GPU (150W) saves ~4x energy")

print("=" * 60)